# 

# Машинное обучение. ВМК МГУ

# Практическое задание 10: Градиентный бустинг деревьев - часть 1

## Уровень: <font color='SkyBlue'>**Базовый (Base)**</font>

# О формате сдачи

🔷 **<font color='plum'>При решении ноутбука используйте данный шаблон</font>**

    ✅ Можно добавлять новые ячейки любых типов
    ❌ Не нужно удалять текстовые ячейки c разметкой частей ноутбука и формулировками заданий


🔷 **<font color='plum'>При оценивании задач учитывается код</font>**

    ✅ Задания, в которых необходим код, обычно помечаются фразами "Your code here"/"Ваш код" и аналогичными
    ❌ Ответы на вопросы без сопутствующего кода оцениваются в 0 баллов
    ❌ Наличе работоспособного кода в ноутбуке, если на сказано иного, обязательно

🔷 **<font color='plum'>При оценивании задач учитываются выводы</font>**

    ✅ Задания, в которых необходимы выводы, обычно помечаются фразами Вывод"/"Ответ на вопрос"/"Ваш текст" и аналогичными
    ✅ Обычно выводы подразумевают под собой текстовый ответ (можно писать markdown, latex).
    ✅ Сопутствующие изображения, графики, таблички - приветствуются!
    ❌ При отсутствии выводов задание не засчитается на полный балл

-----------
<font color="white" style="opacity:0.2025"></font>







__В этом задании вы..:__

- Познакомитесь с несколькими новыми библиотеками машинного обучения
- Сравните между собой разные реализации градиентных бустингов

----

<font color="white" style="opacity:0.2025"></font>
<font color=DarkOrange>**Примерное время выполнения (execution time/время выполнения, если нажать run all) всех ячеек ноутбука при правильной реализации: не более 60 минут </font>**

# Подготовка рабочей среды

Сначала установим нужные нам версии библиотек. Мы гарантируем, что в данных версиях задание будет корректно отрабатывать.

После установки нужных версий, **возможно,** нужно перезагрузить среду (runtime), но скорее всего вам это не понадобится


На скачивание файла и установку понадобится не более 5 минут.

<font color='OrangeRed'>**Важно!**</font>

Устанавливать нужные версии нужно каждый раз, когда создается новый рантайм. Например, если вы 2 часа подряд делаете это задание, то подготовить библиотеки достаточно 1 раз. Но если вы, например, начали в понедельник, затем закрыли/выключили ноутбук, то при продолжении в среду, вам нужно будет запустить рантайм заново и следовательно заново установить библиотеки.

<font color='OrangeRed'>**Важно!**</font>
Если вы предпочитаете делать практические задания на своем личном ноутбуке, то проверьте, что вы установили рабочее окружение в [соответствии с гайдом](https://github.com/MSU-ML-COURSE/ML-COURSE-24-25/blob/main/tutorials/%D0%A2%D1%83%D1%82%D0%BE%D1%80%D0%B8%D0%B0%D0%BB%20%D0%BF%D0%BE%20%D1%83%D1%81%D1%82%D0%B0%D0%BD%D0%BE%D0%B2%D0%BA%D0%B5%20%D1%80%D0%B0%D0%B1%D0%BE%D1%87%D0%B5%D0%B3%D0%BE%20%D0%BE%D0%BA%D1%80%D1%83%D0%B6%D0%B5%D0%BD%D0%B8%D1%8F%20%D0%B2%20Python%20%D0%B4%D0%BB%D1%8F%20%D1%80%D0%B5%D1%88%D0%B5%D0%BD%D0%B8%D1%8F%20%D0%B7%D0%B0%D0%B4%D0%B0%D1%87%20(2).pdf)


In [ ]:
# Установка зависимостей выполняется отдельно по requirements.txt.
# Установка зависимостей выполняется отдельно по requirements.txt.


In [ ]:
import catboost
assert(catboost.__version__ == '1.2.8')

Теперь можно приступать к выполнению задания! :)

-----------
<font color="white" style="opacity:0.2025"></font>




## Введение

Сегодня мы с вами будем решать очень важную задачу, а именно оценивать цену поддержанных автомобилей в ряде стран. Делать мы это будем с помощью различных методов градиентного бустинга. Мы с вами узнаем, что в мире существует не только sklearn, и что существуют библиотеки, облегчающие нам задачу поиска оптимальных параметров для моделей.

<font color='OrangeRed'>**Внимание!**</font> Во всех заданиях в качестве целевой метрики используется MAE (средняя абсолютная ошибка).

Значение MAE вычисляется как

$$
  MAE = \sum_{i = 1}^N\dfrac{|a(x_i) - y_i|}{N},
$$

где $N$ - число объектов в тестовой выборке, $x_i$ - вектор признаков i-го объекта, $a(x_i)$ - предсказание на i-ом объекте, $y_i$ - значение целевого признака на i-м объекте.

## Установка дополнительных библиотек.

В этом задании нам понадобятся три бибиотеки, которыми вы ранее не пользовались в этом курсе, а именно:

<font color='plum'>**XGBoost**:</font> Документация [здесь](https://xgboost.readthedocs.io/en/stable/).<br />
<font color='plum'>**LightGBM**:</font> Документация [здесь](https://lightgbm.readthedocs.io/en/latest/index.html). Также дополнительно про установку [тут](https://pypi.org/project/lightgbm/).<br />
<font color='plum'>**Catboost**:</font> Документация [здесь](https://catboost.ai/en/docs/). Можно найти также некоторую информацию на русском [тут](https://habr.com/ru/company/otus/blog/527554/).<br />
<font color='plum'>**HyperOpt**:</font> Документация [здесь](http://hyperopt.github.io/hyperopt/). <br />


<font color='OrangeRed'>**Внимание!**</font> Вникать и подробно читать документацию к каждой библиотеке нет необходимости! Достаточно обращаться туда для нахождения примеров обучения.


## Подготовка датасета

Все библиотеки, используемые сегодня, мы будем проверять на одних и тех же параметрах:

```n_estimators=1000, max_depth=5, learning_rate=0.1.```

Таким образом мы устанавливаем, соответственно, число деревьев в ансамбле равным 1000, ограничиваем максимальную глубину деревьев 5 и устанавливаем темп обучения равным 0.1. Создадим сразу словарь, чтобы передавать эти параметры создаваемым регрессорам (если вдруг не знали, словарь можно передавать как параметры, поставив перед ним **).

Эти параметры мы вынесем в отдельную переменную `test_parameters`.

При желании можно почитать про этот датасет на платформе kaggle: [ссылка на данные](https://www.kaggle.com/bushnag/cars-in-the-middle-east?select=dataframe_YesIndex_YesHeader_C.csv). <span style="color:red"> <font color="OrangeRed">Не скачивайте датасет из kaggle для выполнения ноутбука, поскольку его могут изменить.</font></span> Нас интересует файл dataframe_YesIndex_YesHeader_C.csv, поскольку он уже хорошо предобработан (хотя, конечно, датасаентисты должны сами уметь это делать, но ладно).
Давайте попробуем загрузить датасет в память и посмотреть, как он выглядит.

При работе в google colab для скачивания датасета достаточно запустить следующую ячейку.

При работе с ноутбуком <span style="color:red"> <font color="OrangeRed">на локальном компьютере</font></span> Вы можете скачать файл по [этой ссылке](https://drive.google.com/file/d/1gdDv2kTCEkF3ia1vvbvRFJM0YfqmPplb/view?usp=sharing) и чуть ниже заменить ` /content/dataframe_YesIndex_YesHeader_C.csv` (в строке с `read_csv`) на ваш локальный путь до файла.

In [ ]:
# Установка зависимостей выполняется отдельно по requirements.txt.
# Установка зависимостей выполняется отдельно по requirements.txt.


In [ ]:
%matplotlib inline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error, make_scorer

from hyperopt import hp, tpe, Trials
from hyperopt.fmin import fmin
from hyperopt.pyll import scope

from xgboost import XGBRegressor

from lightgbm import LGBMRegressor

from catboost import CatBoostRegressor

import matplotlib.pyplot as plt

import pandas as pd

import numpy as np

import time

In [ ]:
test_parameters = {"n_estimators": 1000, "max_depth": 5, "learning_rate":0.1}

# Добавляем команду для скачивания датасета
!gdown --id 1gdDv2kTCEkF3ia1vvbvRFJM0YfqmPplb -O /content/dataframe_YesIndex_YesHeader_C.csv

df = pd.read_csv('data/dataframe_YesIndex_YesHeader_C.csv', index_col=0)
df.head()

### <font color='DarkOrange'>**Задание 0 [без проверки, 0 баллов]**</font>

Посмотрите на цены автомобилей. Попробуйте понять, написаны они в одной валюте или нет. Если нет -- будут ли у нас серьезные проблемы при использовании деревьев? Стоит ли нам что-то сделать для того, чтобы нивелировать эту проблему?


----

В глаза бросаются две проблемы: у нас имеется информация о производителе автомобиля, скрытая в текстовой переменной модели автомобиля (нет отдельного признака). Также имеются некоторые категориальные переменные. Со всем этим безобразием надо что-то сделать.


### <font color='DarkOrange'>**Задание 1 [2 балла]**</font>

**Данные**: датасет со стоимостью подержанных автомобилей  
**Цели**: В данном задании следует выполнить следующие пункты:
1. Изучить датасет, проверить наличие `пропусков.` Под пропусками подразумевается значение N/A вместо признака. При необходимости заменить их на среднее значение признака.
3. Добавить `столбец brand` с информацией о производителе автомобиля (для простоты можно взять первое слово в названии модели).
4. Решить, какие признаки Вы считаете `категориальными.` Конвертировать выбранные категориальные столбцы в тип category.
5. Создать `датасет А`, в котором выбранные категориальные признаки установлены как категориальные. Для этого необходимо создать вектор целевых значений Y (столбец цен автомобилей) и матрицу признаков X, в которой все категориальные переменные помечены как .astype('category'). Дополнительно стоит создать список с названиями и индексами столбцов категориальных переменных (поможет в будущем).
6. Создать `датасет B` без категориальных признаков. Для этого необходимо просто удалить из матрицы признаков все категориальные переменные.
7. Создать `датасет C`, в котором выбранные категориальные признаки закодированы через one-hot encoding. Для этого необходимо из матрицы признаков удалить выбранные категориальные переменные, а затем добавить новые признаки, соответствующие one-hot encoding этих категориальных переменных (со всей этой магией поможет простая функция `pd.get_dummies`).
8. Разбить датасеты на `тренировочное и тестовое множества`, используя `train_test_split(X, y, test_size=0.25, random_state=0)` (зафиксировав random_seed мы получим одинаковое разбиение на обучение/тест для всех трёх выборок).


In [ ]:
datasets = {'A' : None, 'B': None, 'C': None}


# your code here
# 1
print("Количество пропусков в каждом столбце:")
print(df.isnull().sum())
# заменять на среднее не нужно

In [ ]:
# 2
df['brand'] = df['name'].apply(lambda x: str(x).split()[0] if pd.notna(x) else 'Unknown')

df.head()

In [ ]:
# 3
# Обычно категориальные признаки это строковые столбцы
# и столбцы с ограниченным количеством уникальных значений
categorical_columns = []

for col in df.columns:
    if col == 'price':  # Целевая переменная
        continue
    if df[col].dtype == 'object' or df[col].dtype == 'category' or df[col].nunique() < 70: # Поднимал это значение до тех пор пока бренд не стал входить во множество
        categorical_columns.append(col)
        df[col] = df[col].astype('category')

print("Категориальные признаки:")
print(categorical_columns)

In [ ]:
# 4
y = df['price'].values
X_A = df.drop('price', axis=1)

for col in categorical_columns:
    print(X_A[col].name, X_A[col].dtype)

datasets['A'] = {'X': X_A, 'y': y}

In [ ]:
# 5
X_B = df.drop(columns=categorical_columns + ['price'])

for col in X_B.columns:
    print(X_B[col].name)

datasets['B'] = {'X': X_B, 'y': y}

In [ ]:
# 6
X_C_temp = df.drop('price', axis=1)

X_C = pd.get_dummies(X_C_temp, columns=categorical_columns)

datasets['C'] = {'X': X_C, 'y': y}

In [ ]:
# 7
datasets_split = {'A': {}, 'B': {}, 'C': {}}

for name, data in datasets.items():
    X_train, X_test, y_train, y_test = train_test_split(
        data['X'], data['y'], test_size=0.25, random_state=0
    )

    datasets_split[name] = {
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test
    }

Итак, кажется, что у нас всё готово для того, чтобы начать искать ответы на волнующие нас вопросы. Начнем со столь любимой нами библиотеки, а именно...

## Градиентный бустинг со sklearn

Естественно, в `sklearn` имеется реализация градиентного бустинга, которая хранится в `sklearn.ensemble.GradientBoostingRegressor`. Преимущественно данным классом пользуются в учебных заданиях, а в реальных задачах предпочтение отдаётся другим библиотекам. Давайте попробуем понять, заслуженно ли градиентный бустинг в sklearn не пользуется популярностью.

FYI: в sklearn имеется также реализация `GradientBoostingClassifier` для задач классификации, но пользоваться им мы сегодня не будем.

`GradientBoostingRegressor` из коробки не умеет работать с категориальными признаками. Мы к этому уже подготовились, закодировав наши признаки.



### <font color='DarkOrange'>**Задание 2 [1.5 баллa]**</font>

**Данные**: датасет со стоимостью поддержанных автомобилей    
**Метрика**: MAE    
**Цели**: В данном задании следует выполнить следующие пункты:    
1. Обучить `sklearn.ensemble.GradientBoostingRegressor` на `датасетах B и C `(получится два разных регрессора), используя параметры `n_estimators=1000, max_depth=5, learning_rate=0.1` (наши `test_parameters`). Замерить время обучения.
2. Посчитать `MAE` для предсказаний на соответствующих тренировочном и тестовом множествах (можно воспользоваться sklearn.metrics.mean_absolute_error). Замерить время предсказания.
3. Вывести в `таблице DataFrame` результаты о времени обучения, времени предсказания, MAE на тренировочном и тестовом множествах.
<font color='red'>**Внимание!**</font> Для каждого из последующих заданий создавайте новые переменные для таблиц DataFrame - в задании 6 необходимо будет объединить все полученные результаты
4. `Сделайте выводы.` Оцените полезность категориальных переменных. Оцените время, затраченное на обучение. Попробуйте дать оценку получившемуся MAE: оно большое или маленькое?


In [ ]:
df_skl = pd.DataFrame(columns=['Dataset', 'Training time', 'Prediction time', 'Train MAE', 'Test MAE'])

# your cool (yea) code here
for dataset_name in ['B', 'C']:
    X_train = datasets_split[dataset_name]['X_train']
    X_test = datasets_split[dataset_name]['X_test']
    y_train = datasets_split[dataset_name]['y_train']
    y_test = datasets_split[dataset_name]['y_test']

    gb_model = GradientBoostingRegressor(**test_parameters)

    start_time = time.time()
    gb_model.fit(X_train, y_train)
    training_time = time.time() - start_time

    start_time = time.time()
    y_train_pred = gb_model.predict(X_train)
    prediction_time_train = time.time() - start_time

    start_time = time.time()
    y_test_pred = gb_model.predict(X_test)
    prediction_time_test = time.time() - start_time

    total_prediction_time = prediction_time_train + prediction_time_test

    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    new_row = pd.DataFrame({
        'Dataset': [f'Dataset {dataset_name}'],
        'Training time': [f"{training_time:.4f} с"],
        'Prediction time': [f"{total_prediction_time:.4f} с"],
        'Train MAE': [f"{train_mae:.2f}"],
        'Test MAE': [f"{test_mae:.2f}"]
    })

    df_skl = pd.concat([df_skl, new_row], ignore_index=True)

In [ ]:
df_skl.head()

**<font color='MediumOrchid'>Ваши выводы:</font>**

Использование one-hot кодирования (датасет C) существенно повышает качество предсказаний по сравнению с исключением категориальных признаков (датасет B), однако это достигается за счет кратного увеличения времени обучения, что указывает на вычислительную сложность данного метода.

----------------------------------------------
<font color="white" style="opacity:0.2025"></font>

И так, sklearn предлагает нам привычный и простой способ тренировки моделей. Однако, для градиентного бустинга существует множество эвристик и трюков, помогающих улучшить результаты. И одна из библиотек, реализующая улучшенную версию бустинга, называется xgboost.

## Градиентный бустинг с XGBoost

<font color='CornflowerBlue'>**XGBoost**</font> стала достаточно популярной библиотекой, которая позволяет добиться хороших результатов без особых усилий (во всяком случае, так гласят легенды). И у нас для вас две новости: хорошая и плохая. Хорошая – xgboost, хоть и является сторонней библиотекой, сохраняет интерфейс sklearn. Плохая – она тоже не умеет из коробки работать с категориальными признаками. Ну что же, давайте пощупаем это безобразие. На этот раз нас интересует класс xgboost.XGBRegressor.

<font color='MediumTurquoise'>**Daily Fact:**</font> на самом деле в xgboost есть экспериментальная поддержка категориальных переменных, которая еще не добавлена в стабильную версию, поэтому ею мы пользоваться не будем.




### <font color='DarkOrange'>**Задание 3 [1.5 баллa]**</font>

**Данные**: датасет со стоимостью поддержанных автомобилей  
**Метрика**: MAE  
**Цели**: В данном задании следует выполнить следующие пункты:  
1. Обучить `xgboost.XGBRegressor` на `датасетах B и C`, используя параметры ```n_estimators=1000, max_depth=5, learning_rate=0.1``` (наши `test_parameters`). Замерить время обучения.
2. Посчитать `MAE` для полученных предсказаний на соответствующих тренировочном и тестовом множествах (можно воспользоваться `sklearn.metrics.mean_absolute_error`). Замерить время предсказания
3. Вывести в `таблице DataFrame` результаты о времени обучения, времени предсказания, MAE на тренировочном и тестовом множествах.
4. `Сделайте выводы.` Оцените время, затраченное на обучение, сравните с sklearn.

In [ ]:
df_xgb = pd.DataFrame(columns=['Dataset', 'Training time', 'Prediction time', 'Train MAE', 'Test MAE'])

# your perfect ;0 code here
for dataset_name in ['B', 'C']:
    X_train = datasets_split[dataset_name]['X_train']
    X_test = datasets_split[dataset_name]['X_test']
    y_train = datasets_split[dataset_name]['y_train']
    y_test = datasets_split[dataset_name]['y_test']

    xgbr_model = XGBRegressor(**test_parameters)

    start_time = time.time()
    xgbr_model.fit(X_train, y_train)
    training_time = time.time() - start_time

    start_time = time.time()
    y_train_pred = xgbr_model.predict(X_train)
    prediction_time_train = time.time() - start_time

    start_time = time.time()
    y_test_pred = xgbr_model.predict(X_test)
    prediction_time_test = time.time() - start_time

    total_prediction_time = prediction_time_train + prediction_time_test

    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    new_row = pd.DataFrame({
        'Dataset': [f'Dataset {dataset_name}'],
        'Training time': [f"{training_time:.4f} с"],
        'Prediction time': [f"{total_prediction_time:.4f} с"],
        'Train MAE': [f"{train_mae:.2f}"],
        'Test MAE': [f"{test_mae:.2f}"]
    })

    df_xgb = pd.concat([df_xgb, new_row], ignore_index=True)

In [ ]:
df_xgb

**<font color='MediumOrchid'>Ваши выводы:</font>**

Обучение происходит значительно быстрее, чем в Sklearn, особенно на обработанном датасете C. Несмотря на небольшое снижение точности по сравнению со Sklearn, относительная разница в качестве между датасетами B и C сохраняется, что подтверждает пользу категориальных признаков.

---

Таким вот несложным образом нам удалось воспользоваться еще одной библиотекой. Давайте перейдем к третьей библиотеке, которую мы сегодня изучим, и называется она lightgbm.

## Градиентный бустинг в lightgbm

Про <font color='CornflowerBlue'>**Lightgbm**</font> легенды гласят, что она очень быстрая и легкая: что самый большой датасет она без проблем переварит за относительно небольшое время. А еще её разрабатывали не хухры кто, а сами мелкомягкие, так что попробовать её определённо стоит. На этот раз у нас вновь две новости, две хороших и две плохих.

Первая хорошая — lightgbm тоже похож на sklearn. Вторая хорошая — lightgbm умеет из коробки работать с категориальными признаками!

Плохая — делает она это немного нетривиально (не зря я просил вас запомнить категориальные переменные!). Вторая плохая — да, нам придётся этим пользоваться для выполнения задания.

<font color='OrangeRed'>**Внимание!**</font> `LightGBM` может писать сотню радостных сообщений о том, что он увидел ваши categorical_feature и использует их. При желании можете заглушать эти оповещения, ибо в случае перебора параметров их становится слишком много. (заклинание для заглушения: `warnings.filterwarnings("ignore")`)




### <font color='DarkOrange'>**Задание 4 [1.5 баллa]**</font>
**Данные**: датасет со стоимостью поддержанных автомобилей  
**Метрика**: MAE  
**Цели**: В данном задании следует выполнить следующие пункты:  
1. В случае датасета, сохраненном в numpy, lightgbm требует, чтобы категории были закодированы целыми числами от 0 до числа признаков(e.g. ['a', 'b', 'a'] -> [0, 1, 0]). Сделайте это для `датасета A`. Если вы используете pandas, то для датасета A достаточно установить соответствующие столбцы типа "категория", тогда categorical_feature='auto' сам всё подхватит.
2. Обучить `lightgbm.LGBMRegressor` на `датасетах A, B и C`, используя параметры ```n_estimators=1000, max_depth=5, learning_rate=0.1.``` В случае датасета A, передайте в функцию fit индексы/имена категориальных признаков. Замерить время обучения.
4. Посчитать `MAE` для полученных предсказаний на соответствующих тренировочном и тестовом множествах (можно воспользоваться sklearn.metrics.mean_absolute_error).
5. `Сделайте выводы` о полезности использования категориальных переменных и времени обучения.
6. Вывести результаты и время в `таблице DataFrame`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

df_lightgbm = pd.DataFrame(columns=['Dataset', 'Training time', 'Prediction time', 'Train MAE', 'Test MAE'])

# your sweet (fx?) code here
for dataset_name in ['A', 'B', 'C']:
    X_train = datasets_split[dataset_name]['X_train']
    X_test = datasets_split[dataset_name]['X_test']
    y_train = datasets_split[dataset_name]['y_train']
    y_test = datasets_split[dataset_name]['y_test']

    if dataset_name == 'A':
        cat_columns = [col for col in X_train.columns if col in categorical_columns]

        for col in cat_columns:
            X_train[col] = X_train[col].astype('category')
            X_test[col] = X_test[col].astype('category')

        categorical_features = cat_columns

    lgbmr_model = LGBMRegressor(**test_parameters)

    start_time = time.time()
    if dataset_name == 'A':
        lgbmr_model.fit(
            X_train, y_train,
            categorical_feature=categorical_features
        )
    else:
        lgbmr_model.fit(X_train, y_train)
    training_time = time.time() - start_time

    start_time = time.time()
    y_train_pred = lgbmr_model.predict(X_train)
    prediction_time_train = time.time() - start_time

    start_time = time.time()
    y_test_pred = lgbmr_model.predict(X_test)
    prediction_time_test = time.time() - start_time

    total_prediction_time = prediction_time_train + prediction_time_test

    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    new_row = pd.DataFrame({
        'Dataset': [f'Dataset {dataset_name}'],
        'Training time': [f"{training_time:.4f} с"],
        'Prediction time': [f"{total_prediction_time:.4f} с"],
        'Train MAE': [f"{train_mae:.2f}"],
        'Test MAE': [f"{test_mae:.2f}"]
    })

    df_lightgbm = pd.concat([df_lightgbm, new_row], ignore_index=True)

In [ ]:
df_lightgbm

**<font color='MediumOrchid'>Ваши выводы:</font>**

Демонстрирует самую высокую скорость обучения, особенно на необработанных данных. Однако это преимущество в скорости достигается за счет некоторой потери в точности, которая, тем не менее, остается на приемлемом уровне.

----

Как вы видите, иногда можно не возиться с OHE, а позволить библиотекам самим это сделать (хотя иногда при этом приходится повозиться с самими данными, чтобы библиотека их съела).

Итак, мы с вами почти стали мастерами градиентного бустинга. Переходим к последней звезде нашего хит-парада.


## Градиентный бустинг в catboost

_**Пролог:** Why does catboost need random seed? It grows random trees!_

Библиотека, созданная в тёмных подвалах Яндекса. По легендам эта библиотека умеет работать с категориальными данными, быстрая, эффективная, легко настраивается, из коробки понимает текстовые признаки в задачах классификации и спасла Брюса Уиллиса. Давайте проверим. В качестве регрессора нас интересует `catboost.CatBoostRegressor.`

<font color='OrangeRed'>**Внимание!**</font> Эта библиотека еще более болтлива, чем lightgbm, но это (почти) полностью лечится с помощью verbose. Однако ноутбуки всё равно неприлично разрастаются в размерах, поэтому просим вас сделать следующее: <font color='OrangeRed'> обучать catboost в первой ячейке и очистить её вывод перед сдачей. </font> Все необходимые выводы стоит делать во второй ячейке. Также можно воспользоваться clear_output()


А еще эта библиотека может неожиданно создать вам парочку новых папок...




### <font color='DarkOrange'>**Задание 5 [1.5 баллa]**</font>
**Данные**: датасет со стоимостью поддержанных автомобилей  
**Метрика**: MAE  
**Цели**: В данном задании следует выполнить следующие пункты:  
1. Обучить `catboost.CatBoostRegressor` на `датасетах A, B и C`, используя параметры используя параметры ```n_estimators=1000, max_depth=5, learning_rate=0, verbose=False.```
1. В случае `датасета A`, передайте на вход методу fit/конструктору параметр `cat_features`, содержащий имена/индексы категориальных переменных. В данном случае переводить категории в целые числа, как мы делали для lightgbm, не нужно. `Замерьте время обучения.`
3. Посчитать `MAE` для полученных предсказаний на соответствующих тренировочном и тестовом множествах (можно воспользоваться sklearn.metrics.mean_absolute_error).
4. `Сделайте выводы` и полезности использования категориальных переменных и времени обучения.
5. Вывести результаты и время в `таблице DataFrame.`


In [ ]:
from IPython.display import clear_output

df_catboost = pd.DataFrame(columns=['Dataset', 'Training time', 'Prediction time', 'Train MAE', 'Test MAE'])

# your cute code here
params = {
    'n_estimators': 1000,
    'max_depth': 5,
    'learning_rate': 0.1,
    'random_seed': 42,
    'verbose': False,
}

for dataset_name in ['A', 'B', 'C']:
    X_train = datasets_split[dataset_name]['X_train']
    X_test = datasets_split[dataset_name]['X_test']
    y_train = datasets_split[dataset_name]['y_train']
    y_test = datasets_split[dataset_name]['y_test']

    if dataset_name == 'A':
        cat_features = [col for col in X_train.columns if col in categorical_columns]
        for i, col in enumerate(cat_columns):
            X_train[col] = X_train[col].astype(str)
            X_test[col] = X_test[col].astype(str)

    cat = CatBoostRegressor(**params)

    start_time = time.time()
    if dataset_name == 'A':
        cat.fit(
            X_train, y_train,
            cat_features=cat_features,
            verbose=False
        )
    else:
        cat.fit(X_train, y_train, verbose=False)
    training_time = time.time() - start_time

    clear_output()
    #может быть полезно если catboost будет слишком болтливым и неумолкаемым
    #(чтобы не сдавать ноутбуки с кучей ненужного кода)

    start_time = time.time()
    y_train_pred = cat.predict(X_train)
    pred_time_train = time.time() - start_time

    start_time = time.time()
    y_test_pred = cat.predict(X_test)
    pred_time_test = time.time() - start_time

    total_pred_time = pred_time_train + pred_time_test

    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    new_row = pd.DataFrame({
        'Dataset': [f'Dataset {dataset_name}'],
        'Training time': [f"{training_time:.4f} с"],
        'Prediction time': [f"{total_pred_time:.4f} с"],
        'Train MAE': [f"{train_mae:,.2f}"],
        'Test MAE': [f"{test_mae:,.2f}"]
    })

    df_catboost = pd.concat([df_catboost, new_row], ignore_index=True)



In [ ]:
df_catboost

**<font color='MediumOrchid'>Ваши выводы:</font>**

Показывает наилучшую точность предсказаний, особенно на датасетах A и B, с приемлемым временем обучения, занимающим промежуточное положение между XGBoost и LightGBM.

----

И так, мы наконец познакомились со всеми библиотеками градиентного бустинга.

## Обобщение результатов

На текущем этапе у вас должно быть несколько датафреймов результатами по каждой библиотеке. Мы, конечно, сделали некоторые выводы, но пришло время собрать это в красивый отчет.



### <font color='DarkOrange'>**Задание 6 [2 баллa]**</font>
**Данные**: датасет с ценами поддержанных автомобилей  
**Цели**: В данном задании следует выполнить следующие пункты:  
1. При помощи `одного или нескольких графиков` показать результаты различных библиотек: времени работы и качество результатов. Можете воспользоваться любым типом графиков: гистограммы, scatter и т.д. По этим графикам должно быть понятно какая библиотека и насколько быстрее, насколько различается их качество, сравнение оптимизированных и неоптимизированных параметров.
2. По графикам `сравните` библиотеки, производительность и качество работы. Опишите `ваши выводы` ниже.

In [ ]:
# Your great pictures and conclusions below!

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df_skl_anal = df_skl.copy()
df_xgb_anal = df_xgb.copy()
df_lightgbm_anal = df_lightgbm.copy()
df_catboost_anal = df_catboost.copy()

for df in [df_skl_anal, df_xgb_anal, df_lightgbm_anal, df_catboost_anal]:
    for col in ['Training time', 'Prediction time', 'Train MAE', 'Test MAE']:
        df[col] = df[col].astype(str).str.replace(' с', '').str.replace(',', '').astype(float)

libraries = {
    'Sklearn': df_skl_anal,
    'XGBoost': df_xgb_anal,
    'LightGBM': df_lightgbm_anal,
    'CatBoost': df_catboost_anal
}

comparison_data = []

for lib_name, df in libraries.items():
    for _, row in df.iterrows():
        dataset = row['Dataset']
        comparison_data.append({
            'Library': lib_name,
            'Dataset': dataset,
            'Training time': row['Training time'],
            'Prediction time': row['Prediction time'],
            'Train MAE': row['Train MAE'],
            'Test MAE': row['Test MAE']
        })

df_comparison = pd.DataFrame(comparison_data)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Сравнение библиотек градиентного бустинга', fontsize=16, fontweight='bold')
colors = {'Sklearn': '#105775', 'XGBoost': '#17b9ff', 'LightGBM': '#ef4927', 'CatBoost': '#ffcc00'}
datasets = ['Dataset A', 'Dataset B', 'Dataset C']

def create_grouped_bar(ax, metric, title, ylabel):
    x = np.arange(len(datasets))
    width = 0.2
    multiplier = 0

    for lib_name, color in colors.items():
        values = []
        for dataset in datasets:
            mask = (df_comparison['Library'] == lib_name) & (df_comparison['Dataset'] == dataset)
            if mask.any():
                values.append(df_comparison.loc[mask, metric].values[0])
            else:
                values.append(0)

        offset = width * multiplier
        bars = ax.bar(x + offset, values, width, label=lib_name, color=color, alpha=0.8)
        multiplier += 1

        for bar, val in zip(bars, values):
            if val > 0:
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height,
                       f'{val:.1f}' if 'time' in metric else f'{val/1000:.1f}K',
                       ha='center', va='bottom', fontsize=8, rotation=45)

    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(datasets, fontsize=11)
    ax.legend(loc='upper left', fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')

# График Training time
create_grouped_bar(axes[0, 0], 'Training time', 'Время обучения (секунды)', 'Секунды')
# График Prediction time
create_grouped_bar(axes[0, 1], 'Prediction time', 'Время предсказания (секунды)', 'Секунды')
# График Train MAE
create_grouped_bar(axes[1, 0], 'Train MAE', 'Ошибка на тренировочных (MAE)', 'MAE')
# График Test MAE
create_grouped_bar(axes[1, 1], 'Test MAE', 'Ошибка на тестовых (MAE)', 'MAE')

plt.tight_layout()
plt.show()

**<font color='MediumOrchid'>Ваши выводы:</font>**

Наиболее медленной в обучении оказалась библиотека Sklearn, а самой быстрой — LightGBM. По скорости предсказания лучший результат показал CatBoost. По качеству на тестовых данных CatBoost был лучшим на датасетах A и B, в то время как на датасете C с one-hot кодированием неожиданно лучше всех справился Sklearn. При этом LightGBM показал на датасете C более низкую точность по сравнению с остальными.